## 1、Tools 概述

对应《尚硅谷-05-Tools.pdf》第 1 节（PDF 第 1–7 页）。

先看概念，再拆解代码。天气函数都返回**课件中的模拟数据**，不是真实天气。
选择 `langchain` Python 内核；各小节按顺序运行。
标注「需要 API」的单元按需执行，其余示例不调用模型。

### 1.1 工具的重要性

模型负责理解问题、决定调用什么工具；工具负责计算、查询数据库或访问外部 API 等具体操作。

在 LangChain 中，工具可以理解为：**可调用函数 + 名称 + 用途描述 + 输入参数说明**。

### 1.2 工具调用的两种方式

#### 1.2.1 直接调用：先测试工具本身

`@tool` 把普通函数转换成工具对象，之后通过 `.invoke()` 执行。

In [1]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气。"""
    # 模拟数据，未接入天气 API
    return f"{city}晴天，温度 15°C"

result = get_weather.invoke({"city": "北京"})
print(result)
print(type(result))

北京晴天，温度 15°C
<class 'str'>


#### 1.2.2 绑定到模型：由模型决定是否调用（需要 API）

下面沿用之前笔记中的 `BAILIAN_API_KEY`、`BAILIAN_API_URL` 和 `qwen3.8-max`。
`.env` 放在工程根目录，在工程根目录或 `3_Tool` 下运行。
模型名沿用原笔记；真实接口是否可用，需要实际请求确认。

**`bind_tools()` 提供工具描述；模型返回调用请求，工具函数仍由我们的 Python 代码执行。**

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 沿用之前笔记的百炼配置；初始化本身不会发送模型请求
load_dotenv(override=True)
BAILIAN_API_KEY = os.getenv("BAILIAN_API_KEY")
BAILIAN_API_URL = os.getenv("BAILIAN_API_URL")
model = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=BAILIAN_API_KEY,
    base_url=BAILIAN_API_URL,
)

In [ ]:
model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke("北京天气如何？")

if response.tool_calls:
    print("AI 想调用工具：", response.tool_calls)
else:
    print("AI 直接回答：", response.content)

# 到这里还没有执行 get_weather

### 1.3 工具调用的整体流程

1. `model.bind_tools([get_weather])`：向模型提供工具描述。
2. `model_with_tools.invoke(messages)`：得到带有 `tool_calls` 的 `AIMessage`。
3. Python 根据工具名和参数执行函数，得到工具结果。
4. 把工具结果作为 `ToolMessage` 放回消息列表，再请求模型生成回答。

消息顺序：`HumanMessage → AIMessage(tool_calls) → ToolMessage → AIMessage(回答)`。

| 字段 | 含义 |
| --- | --- |
| `name` | 要调用的工具名称 |
| `args` | 传给工具的参数字典，键名必须与函数参数匹配 |
| `id` | 本次工具调用的编号，对应 `ToolMessage.tool_call_id` |
| `type` | LangChain 规范化后的调用类型，完整 ToolCall 使用 `"tool_call"` |

### 1.4 从 Message 流转看工具调用

#### 1.4.1 普通函数：执行函数后，手动构造 ToolMessage

这里**手动构造 AIMessage 来模拟模型的调用请求**，方便离线看清消息流转。
`**tool_call["args"]` 会把 `{"city": "北京"}` 展开为 `city="北京"`。

In [2]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def get_weather(city: str) -> str:
    """获取天气的工具。"""
    return f"{city}天气晴朗~"

messages = [HumanMessage(content="今天北京天气如何？")]

# 离线模拟，不是模型的真实响应
response = AIMessage(
    content="",
    tool_calls=[{
        "name": "get_weather",
        "args": {"city": "北京"},
        "id": "call_weather_01",
        "type": "tool_call",
    }],
)
messages.append(response)

for tool_call in response.tool_calls:
    if tool_call["name"] == "get_weather":
        tool_response = ToolMessage(
            content=get_weather(**tool_call["args"]),
            tool_call_id=tool_call["id"],
            name=tool_call["name"],
        )
        messages.append(tool_response)

for message in messages:
    message.pretty_print()

================================ Human Message =================================

今天北京天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_weather_01)
 Call ID: call_weather_01
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京天气晴朗~


#### 1.4.2 @tool：传入完整 ToolCall，自动得到 ToolMessage

沿用上面模拟的 `response`，这次把整个 `tool_call` 传入 `.invoke()`。

In [3]:
@tool
def get_weather(city: str) -> str:
    """获取天气的工具。"""
    return f"{city}天气晴朗~"

messages = [HumanMessage(content="今天北京天气如何？"), response]

for tool_call in response.tool_calls:
    if tool_call["name"] == "get_weather":
        tool_response = get_weather.invoke(tool_call)
        print(type(tool_response))
        messages.append(tool_response)

for message in messages:
    message.pretty_print()

<class 'langchain_core.messages.tool.ToolMessage'>
================================ Human Message =================================

今天北京天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_weather_01)
 Call ID: call_weather_01
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京天气晴朗~


#### 1.4.3 注意区分两种 invoke 输入

对本例这个返回字符串的工具：只传参数字典，得到字符串；传完整 ToolCall，得到 ToolMessage。

In [4]:
tool_call = response.tool_calls[0]

result1 = get_weather.invoke(tool_call["args"])
result2 = get_weather.invoke(tool_call)

print("只传 args：", type(result1).__name__, result1)
print("传完整 ToolCall：", type(result2).__name__)
print("调用编号对应：", result2.tool_call_id == tool_call["id"])

只传 args： str 北京天气晴朗~
传完整 ToolCall： ToolMessage
调用编号对应： True


#### 1.4.4 完整示例：模型请求 → 执行工具 → 模型回答（需要 API）

先执行 1.2.2 的模型初始化单元。下面把课件中的一次工具往返放在一起；
如果模型直接回答，就直接使用这次响应。

In [ ]:
@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气。"""
    return f"{city}天气晴朗~"  # 模拟数据

model_with_tools = model.bind_tools([get_weather])
messages = [HumanMessage(content="今天北京天气如何？")]

# 第一次请求：由模型决定是否调用工具
response = model_with_tools.invoke(messages)
messages.append(response)

if response.tool_calls:
    for tool_call in response.tool_calls:
        if tool_call["name"] == "get_weather":
            tool_response = get_weather.invoke(tool_call)
            messages.append(tool_response)

    # 第二次请求：让模型根据工具结果生成回答
    final_response = model_with_tools.invoke(messages)
    messages.append(final_response)
else:
    final_response = response

for message in messages:
    message.pretty_print()

### 1.5 记住这几个区别

| 操作 | 实际发生的事 |
| --- | --- |
| `bind_tools()` | 绑定描述，不执行天气查询 |
| `response.tool_calls` | 模型提出的调用请求，不是工具结果 |
| 普通函数 `get_weather(**args)` | 执行 Python 函数，需要手动包装 ToolMessage |
| `@tool` 工具 `.invoke(tool_call)` | 执行工具，并自动关联 tool_call_id |

`AIMessage` 和 `ToolMessage` 都要保留在历史消息中，不能只把天气字符串传回模型。
上面演示一次工具往返；若后续响应仍包含 `tool_calls`，应用还要继续处理。

参考：[LangChain 工具调用](https://docs.langchain.com/oss/python/langchain/models#tool-calling)。